# Adaptive Phase-Field Frontier Comparison

Run `python3 benchmark/frontier_comparison/run_comparison.py --out benchmark/frontier_comparison/out` first. The notebook reads the generated metrics and recreates the three summary plots.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

metrics = json.loads(Path('benchmark/frontier_comparison/out/frontier_metrics.json').read_text())
metrics['context_retention'], metrics['adaptive_compute']['hard_to_easy_steps']

In [ ]:
easy = metrics['adaptive_compute']['easy']
hard = metrics['adaptive_compute']['hard']
reference = metrics['reference_anchor']

plt.figure(figsize=(6, 4))
plt.loglog([easy['estimated_flops_per_query'], hard['estimated_flops_per_query']], [easy['prediction_accuracy_proxy'], hard['prediction_accuracy_proxy']], marker='o', label='Phase mesh proxy')
plt.loglog([reference['dense_flops_per_token'], reference['dense_flops_per_token']], [reference['easy_accuracy_anchor'], reference['hard_accuracy_anchor']], marker='s', label='Dense LLM anchor')
plt.xlabel('Estimated FLOPs/query or dense FLOPs/token')
plt.ylabel('Accuracy / prediction proxy')
plt.ylim(0.85, 1.01)
plt.grid(True, which='both', alpha=0.25)
plt.legend();

In [ ]:
curve = metrics['ram_context_curve']
contexts = [row['context_tokens'] for row in curve]
phase_mb = [row['phase_mesh_bytes'] / 1_000_000 for row in curve]
llm_mb = [row['llama3_8b_int4_kv_bytes'] / 1_000_000 for row in curve]

plt.figure(figsize=(6, 4))
plt.plot(contexts, phase_mb, marker='o', label='Phase mesh state')
plt.plot(contexts, llm_mb, marker='s', label='Dense LLM KV anchor')
plt.xlabel('Context tokens')
plt.ylabel('RAM / state MB')
plt.grid(True, alpha=0.25)
plt.legend();

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(['easy', 'hard'], [easy['steps_used'], hard['steps_used']], color=['#2474a6', '#ba4a3a'])
plt.ylabel('Steps used')
plt.title('Adaptive budget allocation')
plt.grid(True, axis='y', alpha=0.25);